<a href="https://colab.research.google.com/github/aodm26/gpt-oss/blob/main/Copy_of_GPT_OSS_Sentiment_analysis_2_min_and_gui.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# GPT-OSS-20B Sentiment Analysis

**Dataset:** `test_split 1.csv` — 484 financial headlines (pre-labelled)  
**Labels:** `positive` / `negative` / `neutral` (text, lowercase)  
**Label split:** neutral=134 · positive=134 · negative=134

### Speed decisions
| Setting | Value | Reason |
|---|---|---|
| `load_in_4bit` | `True` | ~4× less VRAM for weights |
| `max_seq_length` | `1024` | Halves KV-cache pre-allocation vs 2048 |
| `max_new_tokens` | `96` | Label word + 1-sentence reason fits easily |
| `do_sample` | `False` | Greedy decode — no sampling overhead |
| Batch size | `8` | Batching causes KV-cache OOM on T4 |
| Prompt | Forces single label word on last line | Zero-ambiguity parsing |


## 1 · Install Dependencies

In [ ]:

import os, importlib.util
!pip install --upgrade -qqq uv
if importlib.util.find_spec("torch") is None or "COLAB_" in "".join(os.environ.keys()):
    try: import numpy, PIL; _numpy = f"numpy=={numpy.__version__}"; _pil = f"pillow=={PIL.__version__}"
    except: _numpy = "numpy"; _pil = "pillow"
    !uv pip install -qqq \
        "torch>=2.8.0" "triton>=3.4.0" {_numpy} {_pil} torchvision bitsandbytes "transformers==4.56.2" \
        "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
        "unsloth[base] @ git+https://github.com/unslothai/unsloth" \
        git+https://github.com/triton-lang/triton.git@0add68262ab0a2e33b84524346cb27cbb2787356#subdirectory=python/triton_kernels
elif importlib.util.find_spec("unsloth") is None:
    !uv pip install -qqq unsloth
!uv pip install --upgrade --no-deps transformers==4.56.2 tokenizers trl==0.22.2 unsloth unsloth_zoo


## 2 · Load Model (4-bit)

In [ ]:
from unsloth import FastLanguageModel
import torch

# --- OPTIMIZATIONS FOR A100/L4 ---
# 1. Critical Lever: Set padding to LEFT for batching to work correctly
# Load model and tokenizer
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = "unsloth/gpt-oss-20b-unsloth-bnb-4bit",  # 4-bit — fits T4
    dtype          = torch.bfloat16,
    max_seq_length = 1024,   # small = less KV-cache VRAM
    load_in_4bit   = True,
    full_finetuning= False,
)
tokenizer.padding_side = "left"
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

# 2. Precision Lever: Use BFloat16 for A100 (more stable than FP16)
# This is now handled directly in from_pretrained above.

# 3. Unsloth Inference Optimization
FastLanguageModel.for_inference(model)
print("✓ Optimizations applied: BF16 + Left-Padding.")
print("✓ Model ready.")

## 3 · Load Dataset

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd

df = pd.read_csv('/content/drive/MyDrive/test_split 1.csv')

# Normalise labels to Title case so they match parse_label() output
df['sentiment'] = df['sentiment'].str.strip().str.capitalize()

headlines_list  = df['headline'].tolist()
expected_labels = df['sentiment'].tolist()
n = len(headlines_list)

print(f"Loaded {n} headlines.")
print("Label distribution:")
print(df['sentiment'].value_counts().to_string())
print(f"\nSample headline: {headlines_list[0]}")
print(f"Expected label : {expected_labels[0]}")



In [ ]:
# ── Configuration ────────────────────────────────────────────────────────────
HEADLINE_COL = 'headline'
LABEL_COL    = 'sentiment'   # numeric: 0=Positive, 1=Negative, 2=Neutral
RANDOM_STATE = 43            # change for a different random draw
N_SAMPLES    = 80

# ── Numeric → text label mapping ──────────────────────────────────────────────
LABEL_MAP = {0: 'Positive', 1: 'Negative', 2: 'Neutral'}

# ── Random sample ─────────────────────────────────────────────────────────────
sample_df = df.sample(n=N_SAMPLES, random_state=RANDOM_STATE).reset_index(drop=True)
headlines_list = sample_df[HEADLINE_COL].tolist()

expected_labels = sample_df[LABEL_COL].tolist()

print(f'Randomly sampled {len(headlines_list)} headlines (random_state={RANDOM_STATE}).')
print(f'Expected label distribution:')
import collections
print(dict(collections.Counter(expected_labels)))
print('\nFirst 3 headlines:')
for i, h in enumerate(headlines_list[:3]):
    print(f'  {i+1}. [{expected_labels[i]}] {h[:90]}...')

# ── Save sample to CSV ───────────────────────────────────────────────────────
sample_df.to_csv("/content/random80.csv", index=False, encoding="utf-8")

print(f"Saved sampled dataset to: /content/random80.csv")


## 4 · Label Parser

Extracts the final sentiment word from model output.  
Two-pass: first looks for an explicit `Final sentiment label:` line, then falls back to the last valid label word found anywhere in the response.


In [ ]:
import re

VALID_LABELS = {"Positive", "Negative", "Neutral"}

def parse_reasoning_and_label(text: str):
    if not text:
        return "", "Unknown"

    clean = (
        text.replace("\xa0", " ")
            .replace("**", "")
            .replace("*", "")
            .strip()
    )

    reasoning = ""
    label = "Unknown"
    lines = [l.strip() for l in clean.splitlines() if l.strip()]

    # 1) Standard structured fields
    for line in lines:
        m_reason = re.match(r'(?i)^reasoning\s*:\s*(.+)$', line)
        if m_reason:
            reasoning = m_reason.group(1).strip()

        m_label = re.match(r'(?i)^label\s*:\s*(positive|negative|neutral)\s*$', line)
        if m_label:
            label = m_label.group(1).capitalize()

    # 2) Flexible label patterns anywhere in text
    if label == "Unknown":
        patterns = [
            r'(?i)\blabel\s*:\s*(positive|negative|neutral)\b',
            r'(?i)\bfinal sentiment label\s*:\s*(positive|negative|neutral)\b',
            r'(?i)\bsentiment\s+is\s+(positive|negative|neutral)\b',
            r'(?i)\bsentiment\s*:\s*(positive|negative|neutral)\b',
            r'(?i)\bthe sentiment is (positive|negative|neutral)\b',
        ]

        matches = []
        for pat in patterns:
            matches.extend(re.findall(pat, clean))

        if matches:
            label = matches[-1].capitalize()

    # 3) Last valid label anywhere near the end
    if label == "Unknown":
        all_labels = re.findall(r'(?i)\b(positive|negative|neutral)\b', clean)
        if all_labels:
            label = all_labels[-1].capitalize()

    # 4) If no explicit reasoning field, infer reasoning by removing label phrases
    if not reasoning:
        tmp = clean
        tmp = re.sub(r'(?i)\blabel\s*:\s*(positive|negative|neutral)\b', '', tmp)
        tmp = re.sub(r'(?i)\bfinal sentiment label\s*:\s*(positive|negative|neutral)\b', '', tmp)
        tmp = re.sub(r'(?i)\bsentiment\s+is\s+(positive|negative|neutral)\b', '', tmp)
        tmp = re.sub(r'(?i)\bthe sentiment is (positive|negative|neutral)\b', '', tmp)
        reasoning = tmp.strip(" .:-\n\t")

    return reasoning, label

## 5 · Run Inference

## **Batch-Processed Financial Sentiment Classification via LLM Inference**

---

### **Quick Breakdown**
* **Prompt Engineering:** Uses a strictly defined **System Prompt** to force the model into a structured "Reasoning + Label" output format.
* **Batching for Speed:** Processes headlines in groups of **8** (`BATCH_SIZE`) to maximize GPU utility and decrease total processing time.
* **Efficient Generation:** Employs `torch.inference_mode()` and `use_cache=True` to strip away unnecessary gradients and speed up token generation.
* **Real-time Monitoring:** Tracks accuracy, processing speed per item, and **ETA** (Estimated Time of Arrival) to keep you updated on the script's progress.
* **Validation:** Decodes generated IDs and parses them into a structured dictionary to compare model predictions against expected labels.

In [ ]:
import time
import torch

# Tokenizer setup
tokenizer.padding_side = "left"
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

SYSTEM_PROMPT = """
You are a financial news sentiment classifier.
Classify the headline as Positive, Negative, or Neutral.

Output in exactly 2 lines:
Reasoning: <max 8 words>
Label: <Positive, Negative, or Neutral>

Rules:
- Do not repeat the headline.
- Do not output any extra text.
- The second line must start with 'Label:' exactly.
- Use only one of: Positive, Negative, Neutral.
- Never output 'Unknown'.
""".strip()

BATCH_SIZE = 8
DEBUG = False
MAX_NEW_TOKENS = 24

device = next(model.parameters()).device
results = []
start = time.time()
n = len(headlines_list)

# Optional: sort by headline length to reduce padding waste
order = sorted(range(n), key=lambda i: len(headlines_list[i]))
sorted_headlines = [headlines_list[i] for i in order]
sorted_expected = [expected_labels[i] for i in order]

for batch_start in range(0, n, BATCH_SIZE):
    batch_end = min(batch_start + BATCH_SIZE, n)
    batch_headlines = sorted_headlines[batch_start:batch_end]
    batch_expected = sorted_expected[batch_start:batch_end]

    batch_text = [
        f"{SYSTEM_PROMPT}\n\nHeadline: {headline}"
        for headline in batch_headlines
    ]

    inputs = tokenizer(
        batch_text,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=256,
    ).to(device)

    input_lens = inputs["attention_mask"].sum(dim=1).tolist()

    with torch.inference_mode():
        out_ids = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            use_cache=True,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
            return_dict_in_generate=False,
        )

    batch_results = []
    for j, (headline, exp, input_len) in enumerate(zip(batch_headlines, batch_expected, input_lens)):
        gen_ids = out_ids[j][input_len:]
        response = tokenizer.decode(gen_ids, skip_special_tokens=True).strip()

        reasoning, pred = parse_reasoning_and_label(response)
        pred = str(pred).strip().capitalize()
        exp = str(exp).strip().capitalize()
        correct = pred == exp

        original_idx = order[batch_start + j] + 1

        item = {
            "index": original_idx,
            "headline": headline,
            "expected": exp,
            "predicted": pred,
            "correct": correct,
            "reasoning": reasoning,
            "raw_response": response,
        }
        results.append(item)
        batch_results.append(item)

        if DEBUG and pred == "Unknown":
            print(f"\n[{original_idx}/{n}] RAW RESPONSE ERROR: {repr(response)}")

    done = batch_end
    elapsed = time.time() - start
    rate = elapsed / done
    eta = rate * (n - done)
    batch_correct = sum(r["correct"] for r in batch_results)

    print(
        f"[{done:3d}/{n}] "
        f"batch_acc={batch_correct}/{len(batch_results)} "
        f"| {rate:.2f}s/item | ETA≈{eta/60:.1f}min"
    )

# Restore original order for downstream analysis
results = sorted(results, key=lambda x: x["index"])

print(f"\n✓ Done! Total: {(time.time() - start)/60:.1f} min")

In [ ]:
!pip -q install gradio feedparser pandas

5.1 gui

In [ ]:
import time
import html
import feedparser
import pandas as pd
import gradio as gr
import torch

# -------------------------------------------------------------------
# 1) MODEL/TOKENIZER RUNTIME SETTINGS
# -------------------------------------------------------------------
tokenizer.padding_side = "left"
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

device = next(model.parameters()).device

# Shorter prompt for faster inference while keeping reasoning
FAST_SYSTEM_PROMPT = """
You are a financial news sentiment classifier.

Classify the headline as Positive, Negative, or Neutral.

Output in exactly 2 lines:
Reasoning: <max 8 words>
Label: <Positive, Negative, or Neutral>

Rules:
- Do not repeat the headline.
- Do not output anything else.
- Keep reasoning short.
- The second line must start with Label:
- Never output Unknown.
""".strip()

# -------------------------------------------------------------------
# 2) FREE RSS FEEDS (NO API KEY)
# -------------------------------------------------------------------
RSS_FEEDS = {
    "Google News Business (IE)": "https://news.google.com/rss/headlines/section/topic/BUSINESS?hl=en-IE&gl=IE&ceid=IE:en",
    "Google News World (IE)": "https://news.google.com/rss/headlines/section/topic/WORLD?hl=en-IE&gl=IE&ceid=IE:en",
    "Google News Technology (IE)": "https://news.google.com/rss/headlines/section/topic/TECHNOLOGY?hl=en-IE&gl=IE&ceid=IE:en",
    "Google News Top Stories (IE)": "https://news.google.com/rss?hl=en-IE&gl=IE&ceid=IE:en",
    "BBC Top Stories": "http://feeds.bbci.co.uk/news/rss.xml",
    "BBC Business": "http://feeds.bbci.co.uk/news/business/rss.xml",
}

# -------------------------------------------------------------------
# 3) RSS FETCH
# -------------------------------------------------------------------
def fetch_rss_headlines(feed_name, max_stories):
    url = RSS_FEEDS[feed_name]
    feed = feedparser.parse(url)

    rows = []
    for entry in feed.entries[:int(max_stories)]:
        title = getattr(entry, "title", "").strip()
        link = getattr(entry, "link", "").strip()
        published = getattr(entry, "published", "").strip()

        source_name = feed_name
        if hasattr(entry, "source") and isinstance(entry.source, dict):
            source_name = entry.source.get("title", feed_name) or feed_name

        if title:
            rows.append({
                "source": source_name,
                "headline": title,
                "published": published,
                "url": link,
            })

    return rows

# -------------------------------------------------------------------
# 4) ROBUST PARSER
# -------------------------------------------------------------------
import re

VALID_LABELS = {"Positive", "Negative", "Neutral"}

def parse_reasoning_and_label(text: str):
    if not text:
        return "", "Unknown"

    clean = (
        text.replace("\xa0", " ")
            .replace("**", "")
            .replace("*", "")
            .strip()
    )

    reasoning = ""
    label = "Unknown"
    lines = [l.strip() for l in clean.splitlines() if l.strip()]

    for line in lines:
        m_reason = re.match(r'(?i)^reasoning\s*:\s*(.+)$', line)
        if m_reason:
            reasoning = m_reason.group(1).strip()

        m_label = re.match(r'(?i)^label\s*:\s*(positive|negative|neutral)\s*$', line)
        if m_label:
            label = m_label.group(1).capitalize()

    if label == "Unknown":
        patterns = [
            r'(?i)\blabel\s*:\s*(positive|negative|neutral)\b',
            r'(?i)\bsentiment\s+is\s+(positive|negative|neutral)\b',
            r'(?i)\bsentiment\s*:\s*(positive|negative|neutral)\b',
            r'(?i)\bthe sentiment is (positive|negative|neutral)\b',
        ]
        matches = []
        for pat in patterns:
            matches.extend(re.findall(pat, clean))
        if matches:
            label = matches[-1].capitalize()

    if label == "Unknown":
        all_labels = re.findall(r'(?i)\b(positive|negative|neutral)\b', clean)
        if all_labels:
            label = all_labels[-1].capitalize()

    if not reasoning:
        tmp = clean
        tmp = re.sub(r'(?i)\blabel\s*:\s*(positive|negative|neutral)\b', '', tmp)
        tmp = re.sub(r'(?i)\bsentiment\s+is\s+(positive|negative|neutral)\b', '', tmp)
        tmp = re.sub(r'(?i)\bthe sentiment is (positive|negative|neutral)\b', '', tmp)
        reasoning = tmp.strip(" .:-\n\t")

    return reasoning, label

# -------------------------------------------------------------------
# 5) FAST BATCH INFERENCE
# -------------------------------------------------------------------
def analyze_headlines_batch(headlines, sources=None, batch_size=8, max_new_tokens=24):
    results = []
    sources = sources or [""] * len(headlines)

    # Sort by length to reduce padding waste
    order = sorted(range(len(headlines)), key=lambda i: len(headlines[i]))
    sorted_headlines = [headlines[i] for i in order]
    sorted_sources = [sources[i] for i in order]

    collected = [None] * len(headlines)

    for batch_start in range(0, len(sorted_headlines), batch_size):
        batch_end = min(batch_start + batch_size, len(sorted_headlines))
        batch_headlines = sorted_headlines[batch_start:batch_end]
        batch_sources = sorted_sources[batch_start:batch_end]

        batch_prompts = []
        for h, s in zip(batch_headlines, batch_sources):
            short_h = h[:140]
            prompt = f"{FAST_SYSTEM_PROMPT}\n\n"
            if s:
                prompt += f"Source: {s}\n"
            prompt += f"Headline: {short_h}"
            batch_prompts.append(prompt)

        inputs = tokenizer(
            batch_prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=256
        ).to(device)

        input_lens = inputs["attention_mask"].sum(dim=1).tolist()

        with torch.inference_mode():
            out_ids = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                use_cache=True,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
                return_dict_in_generate=False,
            )

        for j, input_len in enumerate(input_lens):
            gen_ids = out_ids[j][input_len:]
            response = tokenizer.decode(gen_ids, skip_special_tokens=True).strip()
            reasoning, pred = parse_reasoning_and_label(response)

            if pred == "Unknown":
                txt = response.lower()
                if "positive" in txt:
                    pred = "Positive"
                elif "negative" in txt:
                    pred = "Negative"
                elif "neutral" in txt:
                    pred = "Neutral"

            collected[order[batch_start + j]] = {
                "predicted": pred,
                "reasoning": reasoning,
                "raw_response": response,
            }

    return collected

# -------------------------------------------------------------------
# 6) MAIN APP FUNCTION
# -------------------------------------------------------------------
def fetch_and_analyze(feed_name, max_stories):
    start = time.time()

    try:
        stories = fetch_rss_headlines(feed_name, max_stories)

        if not stories:
            empty_df = pd.DataFrame([{
                "source": feed_name,
                "headline": "No stories found.",
                "published": "",
                "predicted": "Unknown",
                "reasoning": "",
                "url": "",
            }])
            return "No stories returned from RSS feed.", empty_df

        headlines = [s["headline"] for s in stories]
        sources = [s["source"] for s in stories]

        preds = analyze_headlines_batch(
            headlines=headlines,
            sources=sources,
            batch_size=8,
            max_new_tokens=24,
        )

        rows = []
        for story, pred in zip(stories, preds):
            rows.append({
                "source": story["source"],
                "headline": story["headline"],
                "published": story["published"],
                "predicted": pred["predicted"],
                "reasoning": pred["reasoning"],
                "url": story["url"],
            })

        df = pd.DataFrame(rows)
        elapsed = time.time() - start
        status = f"Analyzed {len(df)} stories in {elapsed:.1f}s"

        return status, df

    except Exception as e:
        err_df = pd.DataFrame([{
            "source": "ERROR",
            "headline": str(e),
            "published": "",
            "predicted": "Unknown",
            "reasoning": "Callback failed",
            "url": "",
        }])
        return f"Error: {e}", err_df

# -------------------------------------------------------------------
# 7) GRADIO UI
# -------------------------------------------------------------------
with gr.Blocks() as demo:
    gr.Markdown("## Financial Sentiment Sandbox")
    gr.Markdown("Pick a free RSS source, fetch trending stories, and analyze sentiment with reasoning.")

    with gr.Row():
        feed_name = gr.Dropdown(
            choices=list(RSS_FEEDS.keys()),
            value="Google News Business (IE)",
            label="News source"
        )
        max_stories = gr.Slider(
            minimum=1,
            maximum=12,
            step=1,
            value=6,
            label="Number of stories"
        )

    run_btn = gr.Button("Fetch and Analyze", variant="primary")
    status_box = gr.Textbox(label="Status", interactive=False)

    output_table = gr.Dataframe(
        headers=["source", "headline", "published", "predicted", "reasoning", "url"],
        datatype=["str", "str", "str", "str", "str", "str"],
        label="Trending stories sentiment",
        wrap=True
    )

    run_btn.click(
        fn=fetch_and_analyze,
        inputs=[feed_name, max_stories],
        outputs=[status_box, output_table]
    )

demo.queue()
demo.launch(share=True, debug=True)

## 6 · Accuracy & Confusion Matrix

In [ ]:

import pandas as pd
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

results_df = pd.DataFrame(results)
y_true = results_df['expected'].tolist()
y_pred = results_df['predicted'].tolist()

acc       = accuracy_score(y_true, y_pred)
n_correct = results_df['correct'].sum()
n_unknown = (results_df['predicted'] == 'Unknown').sum()

print("=" * 60)
print(f"  Overall Accuracy : {acc:.1%}  ({n_correct}/{len(results_df)} correct)")
print(f"  Unparsed labels  : {n_unknown}")
print("=" * 60)
print()
print(classification_report(
    y_true, y_pred,
    labels=['Positive', 'Negative', 'Neutral'],
    zero_division=0
))

# ── Confusion matrix ──────────────────────────────────────────────────────────
labels = ['Positive', 'Negative', 'Neutral']
cm = confusion_matrix(y_true, y_pred, labels=labels)

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=labels, yticklabels=labels, linewidths=0.5, ax=ax)
ax.set_xlabel('Predicted', fontsize=12)
ax.set_ylabel('Expected',  fontsize=12)
ax.set_title(f'Confusion Matrix  —  Accuracy: {acc:.1%}', fontsize=13)
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150)
plt.show()
print("Saved → confusion_matrix.png")


## 7 · Misclassified Headlines

In [ ]:
wrong_df = results_df[results_df['correct'] == False][
    ['index', 'headline', 'expected', 'predicted', 'reasoning']
].reset_index(drop=True)

print(f"Misclassified: {len(wrong_df)} / {len(results_df)}")
print()
for _, row in wrong_df.iterrows():
    print(f"#{int(row['index']):3d} | Expected={row['expected']:<9s} Predicted={row['predicted']}")
    print(f"       {row['headline'][:110]}")
    print(f"       Reasoning: {row['reasoning'][:120]}")
    print()


## 8 · Save Results

In [ ]:
out = '/content/sentiment_results.csv'
results_df[['index', 'headline', 'expected', 'predicted', 'correct']].to_csv(out, index=False)
print(f"Results saved → {out}")

print("\nLabel distribution comparison:")
comp = pd.DataFrame({
    'Expected': results_df['expected'].value_counts(),
    'Predicted': results_df['predicted'].value_counts(),
}).fillna(0).astype(int)

print(comp)
#print(comp.to_string())
